# K2 End-to-End Inference — ResUNet2D (E2E) → Pore Segmentation (Residual U-Net 2.5D)
Notebook inference **Skema 2 (end-to-end)**. Alur: **RAW (ber-ring + noisy) → E2E restorasi → segmentasi pore**.

Beda dengan K1 (cascade R→NS→B), di sini restorasi hanya **satu model** (`ResidualUNet2D` E2E) yang menghilangkan ring + noise sekaligus, lalu dilanjutkan blok segmentasi.

**Catatan konvensi (penting — verifikasi):**
- **E2E restorasi** memakai checkpoint `ResidualUNet2D` (key `model_state`, global residual), normalisasi **adaptif p1/p99** — identik gaya K1.
- **Segmentasi** memakai `ResidualUNet` (pore-seg notebook), key checkpoint `state_dict`, normalisasi **/255.0** (bukan p1/p99), dan **PORE_CLASS_IDX = 1** (`softmax[:,1]`). Ini BERBEDA dari blok B K1 (yang pore=0). Jangan sampai tertukar.
- Simpan output di skala [0,1] → uint16 penuh (tanpa denormalisasi balik). SSA memakai faktor 1e-6 yang benar.

In [ ]:
!pip install -q tifffile scikit-image openpyxl pandas psutil

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Imports

In [ ]:
import os, gc, time, json, warnings, threading, multiprocessing
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor, as_completed
from datetime import datetime
from contextlib import contextmanager

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

from tifffile import imread, imwrite
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn
from skimage.filters import threshold_otsu
from scipy.ndimage import binary_erosion
import psutil

warnings.filterwarnings('ignore')
CPU_COUNT = multiprocessing.cpu_count()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
print('device:', device, '| CPU cores:', CPU_COUNT)

## 2. Config
**Verifikasi:**
- `E2E_CKPT` = checkpoint model end-to-end (`ResidualUNet2D`, key `model_state`).
- `SEG_CKPT` = checkpoint pore-seg (`ResidualUNet`, key `state_dict`).
- `PORE_CLASS_IDX = 1` (konvensi pore-seg notebook: `softmax[:,1]`).

In [ ]:
class Cfg:
    GDRIVE_ROOT = '/content/drive/MyDrive/Dataset TA'

    # ─── Checkpoint E2E (restorasi ring+noise sekaligus): ResidualUNet2D ─────
    E2E_CKPT = '/content/drive/MyDrive/Alfian_TA/K2_endtoend_resunet2d/resunet2d_K2_best.pth'
    # ─── Checkpoint segmentasi pore: ResidualUNet 2.5D (pore-seg notebook) ───
    SEG_CKPT = f'{GDRIVE_ROOT}/Image Seg Output/pore_output/best_model.pth'

    # ─── Data input (RAW ber-ring + noisy yang akan direstorasi lalu segmentasi) ─
    FILE_RAW   = f'{GDRIVE_ROOT}/317/Bentheimer 317/Bentheimer_grayscale.tif'
    FILE_GT    = f'{GDRIVE_ROOT}/317/Bentheimer 317/Bentheimer_binary.tif'       # mask biner GT (pore>0). None bila tak ada.
    FILE_CLEAN = f'{GDRIVE_ROOT}/317/Bentheimer 317/Bentheimer_clean_lowrot.tif' # GT grayscale bersih. None bila tak ada.

    SLICE_START  = None
    SLICE_END    = None
    SPATIAL_CROP = None

    # ─── Restorasi E2E — IDENTIK dgn training ResUNet2D ──────────────────────
    PATCH_SIZE = 256
    STRIDE     = 96
    USE_AMP    = False         # samakan dgn training (FP32)

    # ─── Segmentasi pore ─────────────────────────────────────────────────────
    SEG_THRESH     = 0.5
    SEG_IN_CH      = 5
    PORE_CLASS_IDX = 1         # <-- pore-seg notebook pakai softmax[:,1] (BERBEDA dari K1)

    # ─── Petrofisika ─────────────────────────────────────────────────────────
    VOXEL_SIZE_UM   = 5.714
    KC_TORTUOSITY   = 2.5
    KC_SHAPE_FACTOR = 2.0

    SAVE_DIR = '/content/drive/MyDrive/Alfian_TA/K2_endtoend_output'

cfg = Cfg()
os.makedirs(cfg.SAVE_DIR, exist_ok=True)

print("="*68); print("  K2 END-TO-END (ResUNet2D + Pore-Seg) — cek file"); print("="*68)
for label, path in [("E2E  ",cfg.E2E_CKPT),("SEG  ",cfg.SEG_CKPT),
                    ("RAW  ",cfg.FILE_RAW),("GTbin",cfg.FILE_GT or '(—)'),
                    ("GTgry",cfg.FILE_CLEAN or '(—)')]:
    st = ("OK" if os.path.isfile(path) else "TIDAK DITEMUKAN") if path and path!='(—)' else "—"
    print(f"  {label}: [{st}]  {path}")

## 3. Model restorasi E2E: ResidualUNet2D

In [ ]:
# ============================================================================
#  Residual U-Net 2D — varian RESTORASI (regresi grayscale->grayscale)
#    - output 1 channel, GLOBAL RESIDUAL: out = net(x) + x
#    - FILTERS [32,64,128,256,512]
#  IDENTIK dengan arsitektur training K2 end-to-end.
# ============================================================================
class ResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch,  out_ch, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_ch)
        self.relu  = nn.ReLU(inplace=True)
        self.shortcut = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, bias=False),
                                       nn.BatchNorm2d(out_ch))
                         if in_ch != out_ch else nn.Identity())
    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + identity)

class EncoderBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = ResidualBlock(in_ch, out_ch)
        self.pool  = nn.MaxPool2d(2, 2)
    def forward(self, x):
        skip = self.block(x)
        return self.pool(skip), skip

class DecoderBlock(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch):
        super().__init__()
        self.up    = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.block = ResidualBlock(in_ch + skip_ch, out_ch)
    def forward(self, x, skip):
        x = self.up(x)
        if x.shape[2:] != skip.shape[2:]:
            x = F.interpolate(x, size=skip.shape[2:], mode='bilinear', align_corners=True)
        return self.block(torch.cat([x, skip], dim=1))

class ResidualUNet2D(nn.Module):
    FILTERS = [32, 64, 128, 256, 512]
    def __init__(self, in_ch=1, out_ch=1, global_residual=True):
        super().__init__()
        f = self.FILTERS
        self.global_residual = global_residual
        self.enc1 = EncoderBlock(in_ch, f[0]); self.enc2 = EncoderBlock(f[0], f[1])
        self.enc3 = EncoderBlock(f[1], f[2]);  self.enc4 = EncoderBlock(f[2], f[3])
        self.bottleneck = ResidualBlock(f[3], f[4])
        self.dec4 = DecoderBlock(f[4], f[3], f[3]); self.dec3 = DecoderBlock(f[3], f[2], f[2])
        self.dec2 = DecoderBlock(f[2], f[1], f[1]); self.dec1 = DecoderBlock(f[1], f[0], f[0])
        self.out  = nn.Conv2d(f[0], out_ch, kernel_size=1)
    def forward(self, x):
        inp = x
        d, s1 = self.enc1(x); d, s2 = self.enc2(d)
        d, s3 = self.enc3(d); d, s4 = self.enc4(d)
        d = self.bottleneck(d)
        d = self.dec4(d, s4); d = self.dec3(d, s3)
        d = self.dec2(d, s2); d = self.dec1(d, s1)
        r = self.out(d)
        y = r + inp if self.global_residual else r
        return y

def count_parameters(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)

## 4. Load checkpoint E2E

In [ ]:
def load_resunet2d(ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    m = ResidualUNet2D(in_ch=1, out_ch=1, global_residual=True).to(device)
    state = ckpt['model_state'] if 'model_state' in ckpt else ckpt
    m.load_state_dict(state); m.eval()
    arch = ckpt.get('arch','?'); ep = ckpt.get('epoch','?')
    print(f"  E2E ResUNet2D loaded: {os.path.basename(ckpt_path)} | arch={arch} epoch={ep} | params={count_parameters(m):,}")
    return m

model_E2E = load_resunet2d(cfg.E2E_CKPT, device)

## 5. Data helpers (load + normalize_volume)

In [ ]:
def load_and_crop_volume(tif_path, crop_size=None, crop_origin=None):
    print(f"  Loading {os.path.basename(tif_path)} ...", end='', flush=True)
    vol = imread(tif_path).astype(np.float32); print(f" shape={vol.shape}")
    if crop_size is not None:
        D,H,W = vol.shape; cs = crop_size
        if crop_origin is not None: x0,y0,z0 = crop_origin
        else: x0=max(0,(W-cs)//2); y0=max(0,(H-cs)//2); z0=max(0,(D-cs)//2)
        x0=min(x0,W-cs); y0=min(y0,H-cs); z0=min(z0,D-cs)
        vol = vol[z0:z0+cs, y0:y0+cs, x0:x0+cs]
    return vol

def normalize_volume(vol, p1=None, p99=None):
    # Normalisasi adaptif p1/p99; return (vol01, p1, p99) utk denorm balik
    if p1  is None: p1  = np.percentile(vol, 1)
    if p99 is None: p99 = np.percentile(vol, 99)
    vn = np.clip((vol - p1)/(p99 - p1 + 1e-8), 0.0, 1.0)
    return vn.astype(np.float32), float(p1), float(p99)

## 6. Model segmentasi pore: ResidualUNet 2.5D + load
**PENTING:** arsitektur & konvensi diambil PERSIS dari pore-seg notebook:
- key checkpoint `state_dict`
- normalisasi input **/255.0** (bukan p1/p99)
- pore = `softmax(logits)[:,1]`

In [ ]:
# ============================================================================
#  Pore-Seg: Residual U-Net 2.5D (Wang et al. 2024) — SEGMENTASI (softmax, 2 kelas)
#  Arsitektur identik notebook ResidualUNet_PoreSegmentation. Blok residual sama
#  strukturnya dgn di atas, tapi ini SEGMENTASI (out=2). Pakai kelas terpisah
#  agar load_state_dict cocok 1:1 dgn checkpoint pore-seg.
# ============================================================================
class _SegResidualBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv1=nn.Conv2d(in_ch,out_ch,3,padding=1,bias=False); self.bn1=nn.BatchNorm2d(out_ch)
        self.conv2=nn.Conv2d(out_ch,out_ch,3,padding=1,bias=False); self.bn2=nn.BatchNorm2d(out_ch)
        self.relu=nn.ReLU(inplace=True)
        self.shortcut=(nn.Sequential(nn.Conv2d(in_ch,out_ch,1,bias=False),nn.BatchNorm2d(out_ch))
                       if in_ch!=out_ch else nn.Identity())
    def forward(self,x):
        idt=self.shortcut(x); o=self.relu(self.bn1(self.conv1(x))); o=self.bn2(self.conv2(o))
        return self.relu(o+idt)
class _SegEnc(nn.Module):
    def __init__(self,i,o): super().__init__(); self.block=_SegResidualBlock(i,o); self.pool=nn.MaxPool2d(2,2)
    def forward(self,x): s=self.block(x); return self.pool(s),s
class _SegDec(nn.Module):
    def __init__(self,i,sk,o):
        super().__init__(); self.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=True)
        self.block=_SegResidualBlock(i+sk,o)
    def forward(self,x,s):
        x=self.up(x)
        if x.shape[2:]!=s.shape[2:]: x=F.interpolate(x,size=s.shape[2:],mode='bilinear',align_corners=True)
        return self.block(torch.cat([x,s],1))
class ResidualUNet(nn.Module):
    FILTERS=[32,64,128,256,512]
    def __init__(self,in_ch=5,n_classes=2):
        super().__init__(); f=self.FILTERS
        self.enc1=_SegEnc(in_ch,f[0]); self.enc2=_SegEnc(f[0],f[1])
        self.enc3=_SegEnc(f[1],f[2]);  self.enc4=_SegEnc(f[2],f[3])
        self.bottleneck=_SegResidualBlock(f[3],f[4])
        self.dec4=_SegDec(f[4],f[3],f[3]); self.dec3=_SegDec(f[3],f[2],f[2])
        self.dec2=_SegDec(f[2],f[1],f[1]); self.dec1=_SegDec(f[1],f[0],f[0])
        self.out=nn.Conv2d(f[0],n_classes,kernel_size=1)
    def forward(self,x):
        x,s1=self.enc1(x); x,s2=self.enc2(x); x,s3=self.enc3(x); x,s4=self.enc4(x)
        x=self.bottleneck(x)
        x=self.dec4(x,s4); x=self.dec3(x,s3); x=self.dec2(x,s2); x=self.dec1(x,s1)
        return self.out(x)

def load_seg(ckpt_path, device, in_ch=5):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    m = ResidualUNet(in_ch=in_ch, n_classes=2).to(device)
    state = ckpt['state_dict'] if 'state_dict' in ckpt else (ckpt.get('model_state', ckpt))
    m.load_state_dict(state); m.eval()
    ep = ckpt.get('epoch','?'); vm = ckpt.get('val_metrics','?')
    print(f"  SEG ResidualUNet loaded: {os.path.basename(ckpt_path)} | epoch={ep} | val_metrics={vm} | params={count_parameters(m):,}")
    return m

seg_model = load_seg(cfg.SEG_CKPT, device, cfg.SEG_IN_CH)

## 7. Inference helpers (patch + Hann untuk restorasi)

In [ ]:
@torch.no_grad()
def infer_slice(model, sl2d, dev, ps=256, stride=96, use_amp=False):
    # Inference 1 slice 2D, patch + Hann blending, output skala [0,1]
    H,W=sl2d.shape; out=np.zeros((H,W),np.float64); wgt=np.zeros((H,W),np.float64)
    g=np.outer(np.hanning(ps),np.hanning(ps))+1e-6
    ys=list(range(0,H-ps+1,stride)); xs=list(range(0,W-ps+1,stride))
    if not ys or ys[-1]+ps<H: ys.append(max(0,H-ps))
    if not xs or xs[-1]+ps<W: xs.append(max(0,W-ps))
    model.eval()
    for y in ys:
        for x in xs:
            t=torch.from_numpy(sl2d[y:y+ps,x:x+ps][None,None]).float().to(dev)
            with torch.autocast('cuda', enabled=use_amp):
                p=model(t).squeeze().float().cpu().numpy()
            out[y:y+ps,x:x+ps]+=p*g; wgt[y:y+ps,x:x+ps]+=g
    return np.clip(out/wgt,0,1).astype(np.float32)

def infer_volume(model, vol01, dev, ps, stride, use_amp, tag='infer'):
    o=np.zeros_like(vol01)
    for z in tqdm(range(vol01.shape[0]), desc=tag):
        o[z]=infer_slice(model, vol01[z], dev, ps, stride, use_amp)
        if z%50==0 and z>0:
            gc.collect()
            if dev.type=='cuda': torch.cuda.empty_cache()
    return o

## 8. Load RAW (+ GT opsional)

In [ ]:
# ===== LOAD RAW + normalisasi adaptif (p1/p99 input sendiri) =====
print("="*68); print("  LOAD RAW"); print("="*68)
raw = load_and_crop_volume(cfg.FILE_RAW, cfg.SPATIAL_CROP, None)
if cfg.SLICE_START is not None or cfg.SLICE_END is not None:
    s = cfg.SLICE_START or 0; e = cfg.SLICE_END if cfg.SLICE_END is not None else raw.shape[0]
    raw = raw[s:e]

vol_raw, raw_p1, raw_p99 = normalize_volume(raw)
n_slices = vol_raw.shape[0]
print(f"  RAW: {vol_raw.shape} range_asli=[{raw.min():.1f},{raw.max():.1f}] p1={raw_p1:.2f} p99={raw_p99:.2f}")
del raw; gc.collect()

# mask GT biner (Dice/IoU)
mask_gt=None
if cfg.FILE_GT and os.path.isfile(cfg.FILE_GT):
    mg=load_and_crop_volume(cfg.FILE_GT, cfg.SPATIAL_CROP, None)
    if cfg.SLICE_START is not None or cfg.SLICE_END is not None:
        s=cfg.SLICE_START or 0; e=cfg.SLICE_END if cfg.SLICE_END is not None else mg.shape[0]; mg=mg[s:e]
    mask_gt=(mg>0.5).astype(np.uint8); print(f"  MASK GT: {mask_gt.shape}"); del mg; gc.collect()

# GT grayscale bersih (PSNR/SSIM) — normalisasi adaptif sendiri
vol_clean=None; has_gt_gray=False
if cfg.FILE_CLEAN and os.path.isfile(cfg.FILE_CLEAN):
    cln=load_and_crop_volume(cfg.FILE_CLEAN, cfg.SPATIAL_CROP, None)
    if cfg.SLICE_START is not None or cfg.SLICE_END is not None:
        s=cfg.SLICE_START or 0; e=cfg.SLICE_END if cfg.SLICE_END is not None else cln.shape[0]; cln=cln[s:e]
    vol_clean,_,_=normalize_volume(cln); has_gt_gray=True
    assert vol_clean.shape==vol_raw.shape
    print(f"  GT grayscale: {vol_clean.shape}"); del cln; gc.collect()
else:
    print("  GT grayscale: (tidak ada) — PSNR/SSIM dilewati")

## 9. Segmentasi pore (Residual U-Net 2.5D)
Input segmentasi memakai konvensi **pore-seg notebook**: stack 2.5D 5-slice dinormalisasi **/255.0**.
Karena hasil restorasi berada di skala [0,1], kita kalikan 255 dulu agar konsisten dengan cara model dilatih (`stack/255.0`).

In [ ]:
@torch.no_grad()
def segment_pore(vol01, model, device, threshold=0.5, pore_idx=1):
    # vol01: volume grayscale skala [0,1]. Model pore-seg dilatih dgn input /255.0,
    # jadi kita rekonstruksi skala 0..255: (vol01*255)/255 == vol01 secara numerik,
    # tapi tetap eksplisit demi kejelasan konvensi.
    n=vol01.shape[0]; masks=np.zeros_like(vol01, dtype=np.uint8)
    vol255 = (np.clip(vol01,0,1)*255.0).astype(np.float32)
    for i in tqdm(range(n), desc='pore-seg', unit='slice'):
        stack=np.stack([vol255[max(0,min(i+off,n-1))] for off in (-2,-1,0,1,2)],axis=0)/255.0
        t=torch.from_numpy(stack).float().unsqueeze(0).to(device)
        prob=F.softmax(model(t),dim=1)[0,pore_idx].cpu().numpy()
        masks[i]=(prob>=threshold).astype(np.uint8)
    return masks

## 10. Fungsi petrofisika (SSA fix 1e-6)

In [ ]:
VOX_M = cfg.VOXEL_SIZE_UM*1e-6
def porosity(b): return {'total_frac':float(b.mean()),'per_slice':b.mean(axis=(1,2))}
def ssa_total(b, vox_m):
    surf=b.astype(int)-binary_erosion(b).astype(int)
    D,H,W=b.shape; sa=surf.sum()*vox_m**2; vol=D*H*W*vox_m**3
    Sv=sa/vol if vol>0 else 0.0
    return {'SSA_m2_cm3':float(Sv*1e-6),'Sv_per_m':float(Sv),'surface_voxels':int(surf.sum())}  # FIX: 1e-6 (1 m3=1e6 cm3)
def ssa_per_slice(b, vox_m):
    Sv=np.zeros(b.shape[0])
    for z in range(b.shape[0]):
        sl=b[z]; surf=sl.astype(int)-binary_erosion(sl).astype(int)
        H,W=sl.shape; vol=H*W*vox_m**3
        Sv[z]=surf.sum()*vox_m**2/vol if vol>0 else 0.0
    return Sv
def kozeny_carman(phi, Sv, tau=2.5, k0=2.0):
    if phi<=0 or phi>=1 or Sv<=0: return float('nan'),float('nan')
    K=phi**3/(k0*tau**2*Sv**2*(1-phi)**2); return float(K),float(K/9.869233e-16)
def dice_iou(pred, ref):
    p,r=pred.astype(bool),ref.astype(bool)
    tp=np.logical_and(p,r).sum(); fp=np.logical_and(p,~r).sum(); fn=np.logical_and(~p,r).sum()
    return float(2*tp/(2*tp+fp+fn+1e-8)), float(tp/(tp+fp+fn+1e-8))
print("petrofisika siap (SSA pakai faktor 1e-6 yang benar)")

## 11. PIPELINE END-TO-END (E2E restorasi → segmentasi pore)
Alur K2: **RAW → E2E → SEG**. Tidak ada renormalisasi antar tahap-restorasi karena hanya satu model.
Renormalisasi adaptif tetap dilakukan sebelum masuk segmentasi (menyamakan skala [0,1] sebelum konversi ke konvensi /255.0).

In [ ]:
t0=time.time()

print("\n[1/2] E2E restorasi (ring + noise sekaligus) ...")
vol_restored = infer_volume(model_E2E, vol_raw, device, cfg.PATCH_SIZE, cfg.STRIDE, cfg.USE_AMP, "E2E: restore")

print("\n[2/2] Segmentasi pore (Residual U-Net 2.5D) ...")
vol_restored_n, _, _ = normalize_volume(vol_restored)     # renormalisasi adaptif sebelum segmentasi
mask_pred = segment_pore(vol_restored_n, seg_model, device, cfg.SEG_THRESH, cfg.PORE_CLASS_IDX)

dt=time.time()-t0; pf=mask_pred.mean()
print(f"\n  Pipeline K2 selesai {dt/60:.1f} mnt | pore_fraction K2 = {pf:.3f}")
if pf>0.65: print("    [warn] pore fraction >0.65 — cek polaritas/PORE_CLASS_IDX vs training pore-seg.")

## 12. Metrik grayscale (vs GT)

In [ ]:
df_signal=None
if has_gt_gray:
    print("="*68); print("  METRIK GRAYSCALE per-slice (vs GT bersih)"); print("="*68)
    from scipy.stats import pearsonr
    def _m(c,x):
        return (psnr_fn(c,x,data_range=1.0), ssim_fn(c,x,data_range=1.0),
                float(np.sqrt(np.mean((c-x)**2))), float(np.mean(np.abs(c-x))),
                float(np.mean(x-c)), float(pearsonr(c.ravel(),x.ravel())[0]))
    rec=[]
    for z in tqdm(range(n_slices), desc='signal'):
        c=vol_clean[z]
        pr=_m(c,vol_raw[z]); pe=_m(c,vol_restored[z])
        rec.append({'slice':z,
            'PSNR_raw':pr[0],'PSNR_E2E':pe[0],
            'SSIM_raw':pr[1],'SSIM_E2E':pe[1],
            'RMSE_raw':pr[2],'RMSE_E2E':pe[2],
            'MAE_raw':pr[3],'MAE_E2E':pe[3],
            'Bias_raw':pr[4],'Bias_E2E':pe[4],
            'Corr_raw':pr[5],'Corr_E2E':pe[5]})
    df_signal=pd.DataFrame(rec)
    _s=lambda c:f"{df_signal[c].mean():.4f}±{df_signal[c].std():.4f}"
    print(f"\n  {'Metrik':<6}{'RAW':>22}{'E2E (restored)':>22}")
    for m in ['PSNR','SSIM','RMSE','MAE','Bias','Corr']:
        print(f"  {m:<6}{_s(m+'_raw'):>22}{_s(m+'_E2E'):>22}")
    df_signal.to_csv(os.path.join(cfg.SAVE_DIR,'K2_metrics_per_slice.csv'),index=False)
    print("  tersimpan: K2_metrics_per_slice.csv")
else:
    print("  GT grayscale tidak ada — metrik dilewati.")

## 13. Petrofisika + vs GT

In [ ]:
por_pred=porosity(mask_pred); ssa_pred=ssa_total(mask_pred,VOX_M)
phi=por_pred['total_frac']
Km2,KmD=kozeny_carman(phi,ssa_pred['Sv_per_m'],cfg.KC_TORTUOSITY,cfg.KC_SHAPE_FACTOR)
print("="*60); print("  PETROFISIKA PREDIKSI K2"); print("="*60)
print(f"  Porositas: {phi*100:.3f} %")
print(f"  SSA      : {ssa_pred['SSA_m2_cm3']:.6f} m2/cm3 (Sv={ssa_pred['Sv_per_m']:.1f}/m)")
print(f"  K (KC)   : {KmD:.4e} mD")

rows=[]
if mask_gt is not None:
    assert mask_gt.shape==mask_pred.shape
    dice,iou=dice_iou(mask_pred,mask_gt)
    por_gt=porosity(mask_gt); ssa_gt=ssa_total(mask_gt,VOX_M); phi_gt=por_gt['total_frac']
    _,KmD_gt=kozeny_carman(phi_gt,ssa_gt['Sv_per_m'],cfg.KC_TORTUOSITY,cfg.KC_SHAPE_FACTOR)
    print("="*60); print("  K2 vs GROUND TRUTH"); print("="*60)
    print(f"  Dice {dice:.4f} | IoU {iou:.4f}")
    print(f"  Porositas GT/pred: {phi_gt*100:.3f}/{phi*100:.3f} % (|galat| {abs(phi-phi_gt)*100:.3f})")
    print(f"  SSA GT/pred: {ssa_gt['SSA_m2_cm3']:.6f}/{ssa_pred['SSA_m2_cm3']:.6f} m2/cm3")
    print(f"  K GT/pred: {KmD_gt:.4e}/{KmD:.4e} mD")
    rows.append({'skema':'K2','Dice':dice,'IoU':iou,
        'porositas_GT_%':phi_gt*100,'porositas_pred_%':phi*100,
        'SSA_GT_m2cm3':ssa_gt['SSA_m2_cm3'],'SSA_pred_m2cm3':ssa_pred['SSA_m2_cm3'],
        'K_GT_mD':KmD_gt,'K_pred_mD':KmD})
else:
    rows.append({'skema':'K2','porositas_pred_%':phi*100,
        'SSA_pred_m2cm3':ssa_pred['SSA_m2_cm3'],'K_pred_mD':KmD})
df=pd.DataFrame(rows); df.to_csv(os.path.join(cfg.SAVE_DIR,'K2_metrics.csv'),index=False)
print("  tersimpan: K2_metrics.csv"); df

## 14. Panel visual

In [ ]:
mid=n_slices//2
panels=[('1. RAW',vol_raw[mid])]
if has_gt_gray: panels.append(('GT grayscale',vol_clean[mid]))
panels+=[('2. Restored (E2E)',vol_restored[mid]),('3. Mask K2 (pore)',mask_pred[mid])]
if mask_gt is not None: panels.append(('Mask GT',mask_gt[mid]))
nc=len(panels); fig,ax=plt.subplots(1,nc,figsize=(4.2*nc,5))
for a,(t,im) in zip(ax,panels): a.imshow(im,cmap='gray'); a.set_title(t); a.axis('off')
plt.tight_layout(); plt.savefig(os.path.join(cfg.SAVE_DIR,'K2_panel.png'),dpi=140); plt.show()

## 15. Simpan (FIX brightness)

In [ ]:
# ===== SIMPAN (skala [0,1] -> u16 penuh, tanpa denorm ke raw) =====
print("="*68); print("  SIMPAN GRAYSCALE + MASK"); print("="*68)
print(f"  Range restored: [{vol_restored.min():.3f},{vol_restored.max():.3f}]  (skala [0,1])")

imwrite(os.path.join(cfg.SAVE_DIR,'K2_1_restored_u16.tif'), (np.clip(vol_restored,0,1)*65535).astype(np.uint16))
imwrite(os.path.join(cfg.SAVE_DIR,'K2_1_restored_float.tif'), vol_restored.astype(np.float32))
imwrite(os.path.join(cfg.SAVE_DIR,'K2_2_mask.tif'), (mask_pred*255).astype(np.uint8))
print("  ✓ restored (u16 + float), mask (pore=255)")
print("\n  Di Fiji: set display range SEMUA jendela ke 0–65535 utk perbandingan brightness jujur.")
print("  Tersimpan di", cfg.SAVE_DIR)